# Example: `shugga` classification and metrics for `LD-waves-exp01` (1993)

This notebook runs the full `shugga` workflow for:

- `SIM_NAME = "LD-waves-exp01"`
- `1993-01-01` to `1993-12-31`
- `FI` on `Tc`

It uses the Python API directly so that the workflow is transparent and easy to modify.


In [1]:
import sys
from pathlib import Path
#####################################################################
# make sure this reflects the correct location of mawsons-chest repo
repo_root = Path.home() / "AFIM" / "src" / "mawsons-chest"
#####################################################################
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
from shuga              import RunSpec, ClassificationSpec, MetricsSpec, ShugaPaths
from shuga.classify     import CICEClassifier
from shuga.metrics      import CICEMetrics
from shuga.core.logging import build_file_logger

In [11]:
import importlib
import shuga.metrics.cice as mc
importlib.reload(mc)
CICEMetrics = mc.CICEMetrics

In [3]:
run          = RunSpec(sim_name="LD-waves-exp01",
                       start_date="1993-01-01",
                       end_date="1993-12-31",
                       hemisphere="SH")
classify     = ClassificationSpec(ice_type="FI",
                                    grid_type="Tc",
                                    ispd_thresh=5e-4,
                                    aice_thresh=0.15,
                                    methods=("raw", "binary-days", "rolling-mean"),
                                    bin_window=11,
                                    bin_min_days=9,
                                    roll_window=15,
                                    uvelE_var="uvelE",
                                    uvelN_var="uvelN",
                                    vvelE_var="vvelE",
                                    vvelN_var="vvelN",)
metrics_spec = MetricsSpec()
paths        = ShugaPaths(run=run, classify=classify)
print("CICE store               :", paths.resolve_cice_store())
print("Static store             :", paths.resolve_static_store())
print("Classification root      :", paths.classification_root_path)
print("Binary-days class store  :", paths.classification_store("binary-days"))
print("Binary-days metrics store:", paths.metrics_store("binary-days"))

CICE store               : /g/data/gv90/da1339/afim_output/LD-waves-exp01/zarr/iceh_daily.zarr
Static store             : /g/data/gv90/da1339/afim_output/LD-waves-exp01/zarr/iceh_static.zarr
Classification root      : /g/data/gv90/da1339/afim_output/LD-waves-exp01/zarr/SH/ispd_thresh_5.0e-4/FI/Tc
Binary-days class store  : /g/data/gv90/da1339/afim_output/LD-waves-exp01/zarr/SH/ispd_thresh_5.0e-4/FI/Tc/bin-win-11_bin-min-09/data.zarr
Binary-days metrics store: /g/data/gv90/da1339/afim_output/LD-waves-exp01/zarr/SH/ispd_thresh_5.0e-4/FI/Tc/bin-win-11_bin-min-09/mets.zarr


In [4]:
logger = build_file_logger("shugga.example", paths.logs_root_path / "notebooks" / "LD-waves-exp01_1993_example.log")
logger.info("Notebook logger ready")

2026-04-06 07:51:45,413 - INFO - Notebook logger ready


## 1. Run classification

This computes:

- raw daily FI mask
- binary-days FI mask
- rolling-mean FI mask


In [5]:
classifier = CICEClassifier(run=run, classify=classify, paths=paths, logger=logger)
class_outputs = classifier.run_methods(overwrite=False)
class_outputs

2026-04-06 07:51:45,431 - INFO - Resolved classification root: /g/data/gv90/da1339/afim_output/LD-waves-exp01/zarr/SH/ispd_thresh_5.0e-4/FI/Tc
2026-04-06 07:51:45,432 - INFO - Classification speed reconstruction mode(s): Tc
2026-04-06 07:51:45,433 - INFO - Resolved CICE store: /g/data/gv90/da1339/afim_output/LD-waves-exp01/zarr/iceh_daily.zarr
2026-04-06 07:51:45,434 - INFO - Resolved static store: /g/data/gv90/da1339/afim_output/LD-waves-exp01/zarr/iceh_static.zarr
2026-04-06 07:51:45,448 - INFO - Opening grouped monthly Zarr between 1993-01-01 and 1993-12-31 (12 groups)
/g/data/xp65/public/apps/med_conda/envs/analysis3-25.12/lib/python3.11/site-packages/distributed/diagnostics/nvml.py:14: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml
2026-04-06 07:51:54,220 - INFO - Merging static variables from: /g/data/gv

{'raw': '/g/data/gv90/da1339/afim_output/LD-waves-exp01/zarr/SH/ispd_thresh_5.0e-4/FI/Tc/raw/data.zarr',
 'binary-days': '/g/data/gv90/da1339/afim_output/LD-waves-exp01/zarr/SH/ispd_thresh_5.0e-4/FI/Tc/bin-win-11_bin-min-09/data.zarr',
 'rolling-mean': '/g/data/gv90/da1339/afim_output/LD-waves-exp01/zarr/SH/ispd_thresh_5.0e-4/FI/Tc/roll-days-15/data.zarr'}

## 2. Run metrics

In [12]:
metrics = CICEMetrics(run=run, classify=classify, metrics=metrics_spec, paths=paths, logger=logger)
for method in classify.methods:
    metrics.compute_metrics(method, overwrite=False)

2026-04-06 07:56:17,130 - INFO - Resolved class store for raw: /g/data/gv90/da1339/afim_output/LD-waves-exp01/zarr/SH/ispd_thresh_5.0e-4/FI/Tc/raw/data.zarr
2026-04-06 07:56:17,132 - INFO - Resolved metrics store for raw: /g/data/gv90/da1339/afim_output/LD-waves-exp01/zarr/SH/ispd_thresh_5.0e-4/FI/Tc/raw/mets.zarr
2026-04-06 07:56:17,134 - INFO - Resolved CICE store: /g/data/gv90/da1339/afim_output/LD-waves-exp01/zarr/iceh_daily.zarr
2026-04-06 07:56:17,136 - INFO - Resolved static store: /g/data/gv90/da1339/afim_output/LD-waves-exp01/zarr/iceh_static.zarr
2026-04-06 07:56:17,154 - INFO - Opening grouped monthly Zarr between 1993-01-01 and 1993-12-31 (12 groups)
2026-04-06 07:56:18,409 - INFO - Merging static variables from: /g/data/gv90/da1339/afim_output/LD-waves-exp01/zarr/iceh_static.zarr
2026-04-06 07:56:24,413 - INFO - Metrics store exists and overwrite=False, skipping: /g/data/gv90/da1339/afim_output/LD-waves-exp01/zarr/SH/ispd_thresh_5.0e-4/FI/Tc/raw/mets.zarr
2026-04-06 07:56:

## 3. Load computed metrics quickly

In [7]:
raw_mets  = metrics.load_metrics("raw")
bin_mets  = metrics.load_metrics("binary-days")
roll_mets = metrics.load_metrics("rolling-mean")

## 4. Inspect time series and regional metrics

In [8]:
ds          = metrics.load_cice(["TLON", "TLAT", "tarea"])
lon, lat    = metrics._detect_lonlat(ds)
area        = metrics._ensure_2d_static(ds["tarea"])
region_mask = metrics._region_mask(area, lon, lat)
print("region cell counts")
print(region_mask.sum(dim=("nj", "ni")).compute())
print("region domain area [10^3 km^2]")
print((region_mask.astype("float32") * area).sum(dim=("nj", "ni")).compute() / 1e9)
fi_any = metrics.load_classification("binary-days").any("time")
lon_fi = lon.where(fi_any)
lat_fi = lat.where(fi_any)
print("FI lon range:", float(lon_fi.min().compute()), float(lon_fi.max().compute()))
print("FI lat range:", float(lat_fi.min().compute()), float(lat_fi.max().compute()))
print(bin_mets["FIA_by_region"].max("time").compute())
print(bin_mets["FIT_by_region"].max("time").compute())
print(bin_mets["FIA_by_region"].sel(time="1993-09-28").compute())
print(bin_mets["FIT_by_region"].sel(time="1993-09-28").compute())

region cell counts
<xarray.DataArray (region: 8)> Size: 64B
array([15648, 14896,  9056, 12592, 13824, 33584, 22208, 22592])
Coordinates:
  * region   (region) object 64B 'DML' 'WIO' 'EIO' 'Aus' 'VOL' 'AS' 'BS' 'WS'
region domain area [10^3 km^2]
<xarray.DataArray (region: 8)> Size: 32B
array([2163.204 , 2171.2837, 1341.1633, 1890.6361, 1747.5372, 4169.2227,
       2875.6753, 2661.8364], dtype=float32)
Coordinates:
  * region   (region) object 64B 'DML' 'WIO' 'EIO' 'Aus' 'VOL' 'AS' 'BS' 'WS'
FI lon range: 0.125 347.625
FI lat range: -78.54263305664062 -60.701900482177734
<xarray.DataArray 'FIA_by_region' (region: 8)> Size: 32B
array([  2.0489872,  51.959743 ,  49.1799   , 100.22924  ,  41.632317 ,
        87.69931  , 123.43613  ,  24.719234 ], dtype=float32)
Coordinates:
  * region   (region) object 64B 'DML' 'WIO' 'EIO' 'Aus' 'VOL' 'AS' 'BS' 'WS'
<xarray.DataArray 'FIT_by_region' (region: 8)> Size: 32B
array([3.6062558, 2.3037808, 2.9770815, 2.5392842, 2.7796688, 2.5630412,
       2.29

## 5. Plot FIP, FIA, and FIT with PyGMT

These methods write figures into the default graphics tree under `/g/data/[PROJECT]/[USER]/GRAPHICAL/AFIM/...`.


In [14]:
fip_path = metrics.plot_fip("binary-days")

2026-04-06 07:57:19,588 - INFO - Plotting FIP for region=DML bounds=[-19.0, 18.0, -80.0, -60.0] projection=S-0.5/-90/20c MC=-0.5 -> /g/data/gv90/da1339/GRAPHICAL/AFIM/LD-waves-exp01/DML/FIP/1993-01-01_1993-12-31_binary-days_FIP.png
2026-04-06 07:57:20,642 - INFO - Plotting FIP for region=WIO bounds=[27.0, 71.0, -80.0, -60.0] projection=S49/-90/20c MC=49.0 -> /g/data/gv90/da1339/GRAPHICAL/AFIM/LD-waves-exp01/WIO/FIP/1993-01-01_1993-12-31_binary-days_FIP.png
2026-04-06 07:57:21,736 - INFO - Plotting FIP for region=EIO bounds=[74.0, 103.0, -80.0, -60.0] projection=S88.5/-90/20c MC=88.5 -> /g/data/gv90/da1339/GRAPHICAL/AFIM/LD-waves-exp01/EIO/FIP/1993-01-01_1993-12-31_binary-days_FIP.png
2026-04-06 07:57:22,823 - INFO - Plotting FIP for region=Aus bounds=[103.0, 146.0, -80.0, -60.0] projection=S124.5/-90/20c MC=124.5 -> /g/data/gv90/da1339/GRAPHICAL/AFIM/LD-waves-exp01/Aus/FIP/1993-01-01_1993-12-31_binary-days_FIP.png
2026-04-06 07:57:23,895 - INFO - Plotting FIP for region=VOL bounds=[146

In [8]:
fia_path = metrics.plot_timeseries("FIA", "binary-days", region="total")
fit_path = metrics.plot_timeseries("FIT", "binary-days", region="total")
fip_path, fia_path, fit_path

basemap [ERROR]: Option -B parsing failure. Correct syntax:

-B Specify both (1) basemap frame settings and (2) axes parameters.
Frame settings are modified via an optional single invocation of -B[<axes>][+b][+g<fill>]⏎
       …[+i[<val>]][+n][+o<lon>/<lat>][+s<subtitle>][+t<title>][+w[<pen>]][+x<fill>][+y<fill>]⏎
       …[+z<fill>]
Axes parameters are specified via one or more invocations of -B[p|s][x|y|⏎
       …z]<intervals>[+a<angle>|n|p][+e[l|u]][+f][+l|L<label>][+p<prefix>][+s|S<secondary_label>]⏎
       …[+u<unit>
<intervals> is composed of concatenated [<type>]<stride>[l|p] sub-strings. See basemap
     documentation for more details and examples of all settings.
basemap [ERROR]: Offending option -BWSen


GMTCLibError: Module 'basemap' failed with status code 72:
basemap [ERROR]: Option -B parsing failure. Correct syntax:
basemap [ERROR]: Offending option -BWSen

## 6. Regional example

Use one of the eight Antarctic sectors: `DML`, `WIO`, `EIO`, `Aus`, `VOL`, `AS`, `BS`, `WS`.


In [ ]:
regional_fia_path = metrics.plot_timeseries("FIA", "binary-days", region="WIO")
regional_fit_path = metrics.plot_timeseries("FIT", "binary-days", region="WIO")

regional_fia_path, regional_fit_path

## 7. Notes

- `load_metrics("raw")`, `load_metrics("binary-days")`, and `load_metrics("rolling-mean")` let you reopen products without recomputing them.
- Classification stores live alongside metrics stores under the same method directories.
- The notebook uses the exact same shared `ShuggaPaths` rules as the PBS and CLI workflows.
